# Basketball Data Cleaning & EDA

This notebook walks through data cleaning and exploratory data analysis (EDA) for three basketball datasets. Each section covers a different dataset, following the same pipeline: import → explore → clean → enrich → visualize.

---

### Datasets

| Dataset | File | Coverage |
|---|---|---|
| **College Basketball** | `cbb.csv` | NCAA Division I teams, multiple seasons |
| **NBA Regular Season** | `nba_team_stats_00_to_23.csv` | NBA team stats from 2000–2023 |
| **NBA Playoffs** | `nba_team_stats_playoffs_00_to_21.csv` | NBA playoff team stats from 2000–2021 |

The central question across all three datasets: **does three-point shooting correlate with winning?**

---

## Part 1: College Basketball EDA

The `cbb.csv` dataset comes from [Kaggle's College Basketball Dataset](https://www.kaggle.com/datasets/andrewsundberg/college-basketball-dataset) and contains per-season statistics for NCAA Division I college basketball teams. Each row represents one team's stats for a given season, including shooting percentages, offensive/defensive ratings, and postseason results.

Key columns:
- `TEAM` / `CONF` / `YEAR` — team identity and season
- `W` / `G` — wins and games played
- `POSTSEASON` — how far the team got (R68, R64, ... Champion, or blank if DNQ)
- `SEED` — tournament seed (if applicable)
- Various shooting stats: `2P_O`, `3P_O`, `FT_O`, etc.

### Import libraries and access the csv

In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv("./College basketball csvs/cbb.csv")

### Exploration: metadata

`describe()` gives us a quick statistical summary — count, mean, std, min/max — for every numeric column. Good for spotting outliers and understanding the scale of each stat before we touch anything.

In [ ]:
df.describe() # Meta Data

### Cleaning: checking for null values

`POSTSEASON` and `SEED` are legitimately empty for teams that didn't make the tournament — these aren't missing data, they're meaningful absences. We'll fill them with descriptive labels instead of dropping rows.

In [ ]:
# Null Values
df.isnull().sum()[df.isnull().sum() > 0]

### Cleaning: filling null values

Teams that didn't qualify for the tournament get `"DNQ"` (Did Not Qualify) in `POSTSEASON` and `"UNRANKED"` in `SEED`. This makes downstream grouping and filtering much cleaner.

In [ ]:
df["POSTSEASON"] = df["POSTSEASON"].fillna("DNQ")
df["SEED"] = df["SEED"].fillna("UNRANKED")

### Structuring: exploring categories

Let's look at the unique conferences and how many teams are in each, and how many teams reached each stage of the postseason. This helps us understand the distribution before we visualize anything.

In [ ]:
df["CONF"].unique()

In [ ]:
df.groupby("CONF")["TEAM"].count() # Teams per conference

In [ ]:
df.groupby("POSTSEASON")["TEAM"].count() # Teams for each postseason stage

### Enriching: deriving new attributes

We create two new columns to make analysis easier:
- `WIN_RATE` — wins divided by games played, giving a normalized 0–1 measure of performance regardless of schedule length
- `IN_PLAYOFFS` — binary flag (1/0) for whether the team made the postseason, useful for aggregation and plotting

In [ ]:
# Data Enriching
df["WIN_RATE"] = (df["W"] / df["G"]) 
df["IN_PLAYOFFS"] = df["POSTSEASON"] != "DNQ"
df["IN_PLAYOFFS"] = df["IN_PLAYOFFS"].astype(int)

### Cleaning: data integrity check

A `WIN_RATE` above 1 is impossible — it would mean a team won more games than they played. We filter those rows out to catch any bad data before it skews our analysis.

In [ ]:
# Get all rows where WIN_RATE > 1
# Data Integrity
df = df[df["WIN_RATE"] <= 1]

### Exploration: graphs

**Scatter — Win Rate vs 2-Point Percentage:** Is there a relationship between how well a team shoots inside the arc and how often they win?

**Line — Playoff Teams by Conference Over Time:** Tracks which conferences consistently send teams to the postseason. Useful for spotting dominant conferences and how their strength has shifted year to year.

In [ ]:
plt.scatter(df["WIN_RATE"],df["2P_O"])

plt.title("Relationship between Winrate and 2 Point Percentage")
plt.xlabel("Win Rate")
plt.ylabel("3 Point Percentage")

In [ ]:
# Suppose you already have
df_grouped = df.groupby(["CONF","YEAR"])["IN_PLAYOFFS"].sum().reset_index()

plt.figure(figsize=(14,8))
sns.lineplot(
    data=df_grouped,
    x="YEAR",
    y="IN_PLAYOFFS",
    hue="CONF",
    marker="o"
)

plt.title("Number of Teams in Playoffs by Conference Over Years")
plt.ylabel("Teams in Playoffs")
plt.xlabel("Year")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')  # move legend outside
plt.tight_layout()
plt.show()

---

## Part 2: NBA Regular Season

The NBA regular season dataset covers team-level stats from **2000 to 2023**. Each row is one team's aggregated stats for a given season — things like shooting percentages, points scored, and minutes played. This lets us look at how offensive strategy (especially 3-point volume) has evolved and how it relates to winning.

Key columns:
- `Team` — franchise name
- `win_percentage` — season win rate (0–1)
- `points`, `field_goals_made/attempted`, `field_goal_percentage` — general scoring
- `three_pointers_made/attempted`, `three_point_percentage` — 3-point shooting
- `free_throws_made/attempted`, `free_throw_percentage` — free throws
- `Min` — total minutes played (pace indicator)

### Import libraries and access the csv

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv('https://raw.githubusercontent.com/vikasvittanala/basketball3021/refs/heads/nbaseason/NBA%20team%20stats%20csvs/nba_team_stats_00_to_23.csv')
df.head()

### Structuring: filtering to relevant columns

The raw dataset has many columns we don't need. We keep only the stats relevant to shooting efficiency and winning, which makes the dataframe easier to work with and the correlation analysis more focused.

In [ ]:
df2 = df[['teamstatspk', 'Team', 'win_percentage', 'Min', 'points', 'field_goals_made', 'field_goals_attempted', 'field_goal_percentage', 'three_pointers_made', 'three_pointers_attempted', 'three_point_percentage', 'free_throws_made', 'free_throw_attempted', 'free_throw_percentage']]
df2.head()

### Enriching: deriving new attributes

Two new metrics to better capture a team's 3-point strategy:
- `three_point_shooting_volume` — what fraction of a team's shot attempts are 3-pointers (how much they *rely* on the 3)
- `three_point_total_contribution` — what fraction of total points actually came from 3-pointers (how much they *benefit* from the 3)

These go beyond raw percentages and capture *how central* the 3-point shot is to a team's offense.

In [ ]:
# 3-point shooting volume: shows what proportion of team's shots are 3-pointers
df2['three_point_shooting_volume'] = df2['three_pointers_attempted'] / df2['field_goals_attempted']

# 3-point total contribution: shows ratio of how many points came from 3-pointers
df2['three_point_total_contribution'] = (df2['three_pointers_made'] * 3) / df2['points']
df2.head()

### Cleaning: checking for null values

A quick sanity check — if any nulls exist we'd need to handle them before analysis. Clean data means we can proceed without imputation.

In [ ]:
print(df2.isnull().values.any())

### Exploration and graphs

**Describe:** Summary stats across all numeric columns to understand ranges and spot anything unusual.

**Unique teams:** Verifies franchise coverage — are all 30 teams represented?

**Scatter — 3-Point % vs Win %:** Does shooting better from 3 actually translate to more wins at the NBA level?

**Correlation heatmap:** A broader view of how all the shooting stats relate to each other and to winning. Helps identify which metrics are redundant and which ones actually drive outcomes.

In [ ]:
df2.describe() # Metadata

In [ ]:
df2["Team"].unique()

In [ ]:
sns.scatterplot(data=df2, x='three_point_percentage', y='win_percentage')
plt.title('3-Point Percentage vs. Win Percentage')
plt.xlabel('3-Point Shooting Percentage')
plt.ylabel('Team Win Percentage')
plt.show()

In [ ]:
cols_of_interest = ['win_percentage', 'points', 'field_goal_percentage', 'three_point_percentage', 'free_throw_percentage', 'three_point_shooting_volume', 'three_point_total_contribution']
corr_submatrix = df2[cols_of_interest].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_submatrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix of Team Statistics')
plt.show()

---

## Part 3: NBA Playoffs

The NBA playoffs dataset covers postseason team stats from **2000 to 2021**. Unlike the regular season, every game in the playoffs is a high-stakes elimination contest — teams are more evenly matched and strategies tend to be more deliberate. This makes it an interesting comparison: does 3-point shooting matter *more* or *less* when the competition intensifies?

The schema mirrors the regular season dataset, so the same cleaning and enrichment steps apply.

### Import libraries and access the csv

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_csv('https://raw.githubusercontent.com/vikasvittanala/basketball3021/refs/heads/nbaseason/NBA%20team%20stats%20csvs/nba_team_stats_00_to_23.csv')
df.head()

### Structuring: filtering to relevant columns

Same column selection as Part 2 — keeping only shooting and winning stats for a focused analysis.

In [ ]:
df2 = df[['teamstatspk', 'Team', 'win_percentage', 'Min', 'points', 'field_goals_made', 'field_goals_attempted', 'field_goal_percentage', 'three_pointers_made', 'three_pointers_attempted', 'three_point_percentage', 'free_throws_made', 'free_throw_attempted', 'free_throw_percentage']]
df2.head()

### Enriching: deriving new attributes

Same as Part 2 — `three_point_shooting_volume` captures how heavily teams lean on the 3-point shot in the playoffs.

In [ ]:
# 3-point shooting volume: shows what proportion of team's shots are 3-pointers
df2['three_point_shooting_volume'] = df2['three_pointers_attempted'] / df2['field_goals_attempted']
df2.head()

### Cleaning: checking for null values

Same null check as Part 2 — confirming the playoff dataset is clean before analysis.

In [ ]:
print(df2.isnull().values.any())

### Exploration

Summary stats and team coverage — same checks as Part 2 but on playoff data. Worth comparing the ranges to the regular season to see if playoff teams skew toward better shooting numbers (they should, since only the top teams qualify).

In [ ]:
df2.describe() # Meta Data

In [ ]:
df2["Team"].unique()

### Graphs

Three views on the 3-point vs winning relationship in the playoffs:

**Scatter — 3-Pointers Attempted vs Win %:** Do teams that *shoot more* 3s win more, or does volume alone not matter?

**Scatter + Regression — 3-Point % vs Win %:** Adds a best-fit line to see the direction and strength of the trend. Compare this slope to the regular season to see if efficiency matters more in the playoffs.

**Box Plot — Win % by 3-Point Efficiency Group:** Splits teams into Low/Medium/High 3-point shooters and compares their win rate distributions. If the High group has a clearly higher median, that's strong evidence that 3-point shooting drives playoff success.

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df2['three_pointers_attempted'], df2['win_percentage'])
plt.xlabel('Three Pointers Attempted')
plt.ylabel('Win Percentage')
plt.title('Three Pointers Attempted vs Win Percentage')
plt.show()

In [ ]:
x = df2['three_point_percentage']
y = df2['win_percentage']

plt.figure(figsize=(8,5))
plt.scatter(x,y)

m, b = np.polyfit(x,y,1)
plt.plot(x, m*x + b)
plt.title('Three Point Percentage vs Win Percentage')
plt.xlabel('Three Point Percentage')
plt.ylabel('Win Percentage')
plt.show()

In [ ]:
df2['three_pt_pct_group'] = pd.qcut(df2['three_point_percentage'], 3, labels=['Low', 'Medium', 'High'])

plt.figure(figsize=(8,5))

plt.boxplot([df2[df2['three_pt_pct_group'] == g]['win_percentage'] for g in ['Low', 'Medium', 'High']], tick_labels =['Low', 'Medium', 'High'])
plt.title(' Win Percentage by 3-Point Efficieny')
plt.xlabel('3-Point Percentage Level')
plt.ylabel('Win Percentage')
plt.show()